# Notebook 3 — Coder TD-Learning de zéro

**Chapitre 17 — Monte Carlo, TD-Learning, TD(λ), SARSA et Q-Learning**

**Vous allez implémenter :**
1. **TD(0)** sur un GridWorld minimal.
2. **TD(n)** avec n variable (1, 3, 5).
3. **TD(λ)** avec traces d'éligibilité.
4. **SARSA** sur FrozenLake.
5. **Q-Learning** sur FrozenLake.
6. **Comparaison SARSA vs Q-Learning** sur Cliff Walking.

Équations de référence : TD(0) Éq.2, TD(n) Éq.3, TD(λ) backward Éq.5, SARSA Éq.6, Q-Learning Éq.7.

> Ce notebook est **autonome** : environnements (FrozenLake, CliffWalking) réimplémentés en pur Python/NumPy pour fonctionner sans installation. Seuls `numpy` et `matplotlib` sont requis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

rng = np.random.default_rng(0)
print("Imports OK")

## 0. GridWorld minimal (pour TD(0), TD(n), TD(λ))

Grille `n x n`, départ `(0,0)`, objectif en bas à droite. Coût `-1` par pas, `0` à l'objectif.

In [ ]:
class GridWorld:
    def __init__(self, n=4):
        self.n = n
        self.goal = (n - 1, n - 1)
        self.actions = [0, 1, 2, 3]  # haut, bas, gauche, droite
        self.reset()

    def reset(self):
        self.state = (0, 0)
        return self.state

    def step(self, action):
        r, c = self.state
        if action == 0:   r = max(0, r - 1)
        elif action == 1: r = min(self.n - 1, r + 1)
        elif action == 2: c = max(0, c - 1)
        elif action == 3: c = min(self.n - 1, c + 1)
        self.state = (r, c)
        done = self.state == self.goal
        return self.state, (0.0 if done else -1.0), done

    def tous_les_etats(self):
        return [(r, c) for r in range(self.n) for c in range(self.n)]

def politique_aleatoire(state, env):
    return int(rng.choice(env.actions))

env = GridWorld(n=4)
print("GridWorld prêt. Objectif :", env.goal)

## 1. TD(0)

`V(S_t) <- V(S_t) + alpha [R_{t+1} + gamma V(S_{t+1}) - V(S_t)]`. Mise à jour **à chaque pas** (bootstrap).

In [ ]:
def td_zero(env, politique, n_episodes=5000, alpha=0.1, gamma=0.9):
    V = defaultdict(float)
    for _ in range(n_episodes):
        state = env.reset()
        for _ in range(200):
            action = politique(state, env)
            next_state, reward, done = env.step(action)
            cible = reward + gamma * (0.0 if done else V[next_state])
            V[state] += alpha * (cible - V[state])
            state = next_state
            if done:
                break
    return V

V_td0 = td_zero(env, politique_aleatoire)
print("V(0,0) =", round(V_td0[(0, 0)], 2))

## 2. TD(n) — n pas (1, 3, 5)

`G_t^(n) = R_{t+1} + gamma R_{t+2} + ... + gamma^{n-1} R_{t+n} + gamma^n V(S_{t+n})`.

In [ ]:
def td_n(env, politique, n=3, n_episodes=5000, alpha=0.1, gamma=0.9):
    V = defaultdict(float)
    for _ in range(n_episodes):
        states, rewards = [env.reset()], [0.0]
        T = float("inf")
        t = 0
        while True:
            if t < T:
                action = politique(states[t], env)
                next_state, reward, done = env.step(action)
                states.append(next_state)
                rewards.append(reward)
                if done:
                    T = t + 1
            tau = t - n + 1  # état dont on met à jour la valeur
            if tau >= 0:
                G = 0.0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                if tau + n < T:
                    G += (gamma ** n) * V[states[tau + n]]
                V[states[tau]] += alpha * (G - V[states[tau]])
            if tau == T - 1:
                break
            t += 1
    return V

for n in (1, 3, 5):
    Vn = td_n(env, politique_aleatoire, n=n)
    print(f"TD(n={n}) : V(0,0) = {Vn[(0,0)]:.2f}")

## 3. TD(λ) avec traces d'éligibilité (vue backward)

`e_t(s) = gamma*lambda*e_{t-1}(s) + 1{s = S_t}` puis `V(s) <- V(s) + alpha * delta_t * e_t(s)`.

In [ ]:
def td_lambda(env, politique, lam=0.5, n_episodes=5000, alpha=0.1, gamma=0.9):
    V = defaultdict(float)
    for _ in range(n_episodes):
        e = defaultdict(float)  # traces d'éligibilité
        state = env.reset()
        for _ in range(200):
            action = politique(state, env)
            next_state, reward, done = env.step(action)
            delta = reward + gamma * (0.0 if done else V[next_state]) - V[state]
            e[state] += 1.0
            for s in list(e.keys()):
                V[s] += alpha * delta * e[s]
                e[s] *= gamma * lam
            state = next_state
            if done:
                break
    return V

for lam in (0.0, 0.5, 0.9):
    Vl = td_lambda(env, politique_aleatoire, lam=lam)
    print(f"TD(lambda={lam}) : V(0,0) = {Vl[(0,0)]:.2f}")
print("\nlambda=0 -> proche de TD(0) ; lambda->1 -> proche de Monte Carlo.")

## 4. & 5. SARSA et Q-Learning sur FrozenLake

FrozenLake 4x4 (version déterministe) : `S` départ, `F` glace, `H` trou (échec), `G` objectif (+1).

In [ ]:
class FrozenLake:
    """Version déterministe, compatible avec l'esprit de Gym FrozenLake-v1 (is_slippery=False)."""
    MAP = ["SFFF", "FHFH", "FFFH", "HFFG"]

    def __init__(self):
        self.n = 4
        self.nS = self.n * self.n
        self.nA = 4  # 0=gauche,1=bas,2=droite,3=haut
        self.holes = {r * self.n + c for r, row in enumerate(self.MAP)
                      for c, ch in enumerate(row) if ch == "H"}
        self.goal = self.n * self.n - 1
        self.reset()

    def reset(self):
        self.s = 0
        return self.s

    def step(self, a):
        r, c = divmod(self.s, self.n)
        if a == 0:   c = max(0, c - 1)
        elif a == 1: r = min(self.n - 1, r + 1)
        elif a == 2: c = min(self.n - 1, c + 1)
        elif a == 3: r = max(0, r - 1)
        self.s = r * self.n + c
        if self.s in self.holes:
            return self.s, 0.0, True
        if self.s == self.goal:
            return self.s, 1.0, True
        return self.s, 0.0, False

lake = FrozenLake()
print("FrozenLake prêt. Trous :", sorted(lake.holes), "| Objectif :", lake.goal)

In [ ]:
def eps_greedy(Q, s, nA, epsilon):
    if rng.random() < epsilon:
        return int(rng.integers(nA))
    return int(np.argmax(Q[s]))

def sarsa(env, n_episodes=5000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = np.zeros((env.nS, env.nA))
    for _ in range(n_episodes):
        s = env.reset()
        a = eps_greedy(Q, s, env.nA, epsilon)
        for _ in range(100):
            s2, r, done = env.step(a)
            a2 = eps_greedy(Q, s2, env.nA, epsilon)
            # SARSA : utilise l'action a2 réellement choisie (on-policy)
            cible = r + gamma * (0.0 if done else Q[s2, a2])
            Q[s, a] += alpha * (cible - Q[s, a])
            s, a = s2, a2
            if done:
                break
    return Q

def q_learning(env, n_episodes=5000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = np.zeros((env.nS, env.nA))
    for _ in range(n_episodes):
        s = env.reset()
        for _ in range(100):
            a = eps_greedy(Q, s, env.nA, epsilon)
            s2, r, done = env.step(a)
            # Q-Learning : utilise max_a' Q(s2, a') (off-policy)
            cible = r + gamma * (0.0 if done else np.max(Q[s2]))
            Q[s, a] += alpha * (cible - Q[s, a])
            s = s2
            if done:
                break
    return Q

Q_sarsa = sarsa(lake, n_episodes=20000, epsilon=0.3)
Q_qlear = q_learning(lake, n_episodes=20000, epsilon=0.3)
print("SARSA et Q-Learning entraînés sur FrozenLake.")

In [ ]:
fleches = {0: "<", 1: "v", 2: ">", 3: "^"}

def afficher_politique(Q, env, titre):
    print(titre)
    for r in range(env.n):
        ligne = ""
        for c in range(env.n):
            s = r * env.n + c
            if s in env.holes:
                ligne += " H "
            elif s == env.goal:
                ligne += " G "
            else:
                ligne += f" {fleches[int(np.argmax(Q[s]))]} "
        print(ligne)
    print()

afficher_politique(Q_sarsa, lake, "Politique SARSA :")
afficher_politique(Q_qlear, lake, "Politique Q-Learning :")

## 6. Comparaison SARSA vs Q-Learning sur Cliff Walking

Grille 4x12. Ligne du bas = falaise (`-100` et retour au départ) sauf départ `S` et objectif `G`. Coût `-1` par pas.

**Résultat attendu (cf. Sutton & Barto) :** Q-Learning apprend le **chemin optimal le long de la falaise** (risqué), SARSA apprend un **chemin plus sûr** (s'éloigne du bord) car il intègre l'exploration epsilon.

In [ ]:
class CliffWalking:
    def __init__(self):
        self.rows, self.cols = 4, 12
        self.nS = self.rows * self.cols
        self.nA = 4  # 0=gauche,1=bas,2=droite,3=haut
        self.start = (3, 0)
        self.goal = (3, 11)
        self.reset()

    def idx(self, rc):
        return rc[0] * self.cols + rc[1]

    def reset(self):
        self.pos = self.start
        return self.idx(self.pos)

    def step(self, a):
        r, c = self.pos
        if a == 0:   c = max(0, c - 1)
        elif a == 1: r = min(self.rows - 1, r + 1)
        elif a == 2: c = min(self.cols - 1, c + 1)
        elif a == 3: r = max(0, r - 1)
        self.pos = (r, c)
        # falaise : ligne du bas, colonnes 1..10
        if r == 3 and 1 <= c <= 10:
            self.pos = self.start
            return self.idx(self.pos), -100.0, False
        if self.pos == self.goal:
            return self.idx(self.pos), -1.0, True
        return self.idx(self.pos), -1.0, False

def entrainer(env, algo, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((env.nS, env.nA))
    recompenses = []
    for _ in range(n_episodes):
        s = env.reset()
        total = 0.0
        if algo == "sarsa":
            a = eps_greedy(Q, s, env.nA, epsilon)
        for _ in range(500):
            if algo == "qlearning":
                a = eps_greedy(Q, s, env.nA, epsilon)
            s2, r, done = env.step(a)
            total += r
            if algo == "sarsa":
                a2 = eps_greedy(Q, s2, env.nA, epsilon)
                cible = r + gamma * (0.0 if done else Q[s2, a2])
                Q[s, a] += alpha * (cible - Q[s, a])
                s, a = s2, a2
            else:
                cible = r + gamma * (0.0 if done else np.max(Q[s2]))
                Q[s, a] += alpha * (cible - Q[s, a])
                s = s2
            if done:
                break
        recompenses.append(total)
    return Q, recompenses

cliff = CliffWalking()
Q_s, rec_s = entrainer(cliff, "sarsa")
Q_q, rec_q = entrainer(cliff, "qlearning")
print("Entraînement Cliff Walking terminé.")

In [ ]:
def moyenne_glissante(x, k=10):
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(moyenne_glissante(rec_s), label="SARSA (plus sûr)")
plt.plot(moyenne_glissante(rec_q), label="Q-Learning (optimal risqué)")
plt.xlabel("Épisode")
plt.ylabel("Récompense par épisode (moy. glissante)")
plt.title("Cliff Walking : SARSA vs Q-Learning")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

fleches = {0: "<", 1: "v", 2: ">", 3: "^"}
def afficher_chemin(Q, env, titre):
    print(titre)
    for r in range(env.rows):
        ligne = ""
        for c in range(env.cols):
            if (r, c) == env.start: ligne += " S "
            elif (r, c) == env.goal: ligne += " G "
            elif r == 3 and 1 <= c <= 10: ligne += " C "
            else: ligne += f" {fleches[int(np.argmax(Q[r*env.cols+c]))]} "
        print(ligne)
    print()

afficher_chemin(Q_s, cliff, "Chemin SARSA (s'éloigne de la falaise C) :")
afficher_chemin(Q_q, cliff, "Chemin Q-Learning (longe la falaise C) :")

## 7. À vous de jouer

1. Faites varier `alpha`, `gamma`, `epsilon`, `lambda` et observez l'impact (vitesse, stabilité).
2. Comparez TD(0) / TD(n) / TD(λ) : lequel converge le plus vite sur GridWorld ?
3. Sur Cliff Walking, baissez `epsilon` vers 0 : SARSA finit par rejoindre la politique de Q-Learning. Pourquoi ?
4. **Question finale :** sur quel type d'environnement utiliseriez-vous chaque méthode ? Justifiez en vous appuyant sur les études de cas du chapitre (sections 7, 8 et 16).